# Build Remaining 3 Grain-Based Tables
**MedTrack_DV — Milestone 1, Step: Department Analytics, Resource Utilization, Patient Flow**

Builds on `hospital_overview_dataset.csv` (already saved) plus HMIS `ward`, `bed`, `staff_assignment`, `employee`, `department`.

**Documented limitations (per mentor's 'do not fabricate' rule):**
- `staff_assignment`/`bed` have no date field in HMIS — they are current-state snapshots, not daily logs. Staff Utilization is therefore a **snapshot**, not a daily trend.
- HMIS has no ward-transfer log — each admission has exactly one ward/bed. Patient Flow is modeled as a 2-point event (admission, discharge), not a multi-step journey.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

HMIS_DIR = "../data/raw/hmis"
PROCESSED_DIR = "../data/processed"

hospital_overview = pd.read_csv(f"{PROCESSED_DIR}/hospital_overview_dataset.csv",
                                 parse_dates=['admission_date', 'discharge_date'])
ward = pd.read_csv(f"{HMIS_DIR}/ward.csv")
bed = pd.read_csv(f"{HMIS_DIR}/bed.csv")
department = pd.read_csv(f"{HMIS_DIR}/department.csv")
staff_assignment = pd.read_csv(f"{HMIS_DIR}/staff_assignment.csv")
employee = pd.read_csv(f"{HMIS_DIR}/employee.csv")

print("hospital_overview:", hospital_overview.shape)

hospital_overview: (45000, 30)


## 1. Department Analytics Dataset
**Grain: one row = department + day.** Built by grouping admissions by `department_id` and `admission_date`.

In [2]:
daily_admissions = (
    hospital_overview
    .groupby(['department_id', 'department_name', hospital_overview['admission_date'].dt.date])
    .agg(
        admissions_count=('admission_id', 'count'),
        avg_length_of_stay_days=('length_of_stay_days', 'mean'),
        readmissions_count=('is_readmission', 'sum')
    )
    .reset_index()
    .rename(columns={'admission_date': 'date'})
)

daily_discharges = (
    hospital_overview
    .groupby(['department_id', hospital_overview['discharge_date'].dt.date])
    .agg(discharges_count=('admission_id', 'count'))
    .reset_index()
    .rename(columns={'discharge_date': 'date'})
)

department_analytics = daily_admissions.merge(
    daily_discharges, on=['department_id', 'date'], how='outer'
)

department_analytics['admissions_count'] = department_analytics['admissions_count'].fillna(0)
department_analytics['discharges_count'] = department_analytics['discharges_count'].fillna(0)
department_analytics['readmission_rate_pct'] = (
    department_analytics['readmissions_count'] / department_analytics['admissions_count'].replace(0, np.nan) * 100
).fillna(0).round(2)

print("department_analytics shape:", department_analytics.shape)
department_analytics.head()

department_analytics shape: (13121, 8)


,department_id,department_name,date,admissions_count,avg_length_of_stay_days,readmissions_count,discharges_count,readmission_rate_pct
0,1,Emergency,2020-01-01,6.0,5.000000,0.0,0.0,0.0
1,1,Emergency,2020-01-02,6.0,4.000000,0.0,1.0,0.0
2,1,Emergency,2020-01-03,3.0,4.333333,0.0,3.0,0.0
3,1,Emergency,2020-01-04,1.0,2.000000,0.0,2.0,0.0
4,1,Emergency,2020-01-05,4.0,6.500000,0.0,2.0,0.0


In [3]:
department_analytics.to_csv(f"{PROCESSED_DIR}/department_analytics_dataset.csv", index=False)
print("Saved:", f"{PROCESSED_DIR}/department_analytics_dataset.csv")

Saved: ../data/processed/department_analytics_dataset.csv


## 2. Resource Utilization Dataset — Beds (daily occupancy)
**Grain: one row = department + day + resource_type (bed).**
Derived by exploding each admission's `admission_date` → `discharge_date` range into one row per active day, then counting occupied beds per department per day. Total capacity comes from `ward.total_beds` summed per department.

In [4]:
# Explode each admission into one row per day it was active (occupied a bed)
rows = []
for _, r in hospital_overview[['department_id', 'admission_date', 'discharge_date']].iterrows():
    for day in pd.date_range(r['admission_date'], r['discharge_date'], freq='D'):
        rows.append((r['department_id'], day.date()))

daily_occupied = pd.DataFrame(rows, columns=['department_id', 'date'])
bed_occupancy = (
    daily_occupied.groupby(['department_id', 'date'])
    .size()
    .reset_index(name='occupied_beds_count')
)

print("Row explosion complete:", len(rows), "day-admission records ->", bed_occupancy.shape, "department-day rows")
bed_occupancy.head()

Row explosion complete: 276975 day-admission records -> (13210, 3) department-day rows


,department_id,date,occupied_beds_count
0,1,2020-01-01,6
1,1,2020-01-02,12
2,1,2020-01-03,14
3,1,2020-01-04,12
4,1,2020-01-05,14


In [5]:
# Total bed capacity per department (from ward.total_beds)
ward_capacity = ward.merge(department, on='department_id', how='left')
dept_bed_capacity = ward_capacity.groupby(['department_id', 'department_name'])['total_beds'].sum().reset_index()
dept_bed_capacity = dept_bed_capacity.rename(columns={'total_beds': 'total_beds_capacity'})

resource_utilization_beds = bed_occupancy.merge(dept_bed_capacity, on='department_id', how='left')
resource_utilization_beds['resource_type'] = 'bed'
resource_utilization_beds['utilization_rate_pct'] = (
    resource_utilization_beds['occupied_beds_count'] / resource_utilization_beds['total_beds_capacity'] * 100
).round(2)

print(resource_utilization_beds.shape)
resource_utilization_beds.head()

(13210, 7)


,department_id,date,occupied_beds_count,department_name,total_beds_capacity,resource_type,utilization_rate_pct
0,1,2020-01-01,6,Emergency,75,bed,8.00
1,1,2020-01-02,12,Emergency,75,bed,16.00
2,1,2020-01-03,14,Emergency,75,bed,18.67
3,1,2020-01-04,12,Emergency,75,bed,16.00
4,1,2020-01-05,14,Emergency,75,bed,18.67


## 3. Resource Utilization Dataset — Staff (SNAPSHOT, documented limitation)
`staff_assignment` has no date field — it is a current roster, not a daily log. This is added as a **separate snapshot table**, not merged into the daily bed records, so the difference in granularity is never hidden or faked.

In [6]:
staff_with_dept = staff_assignment.merge(employee, on='employee_id', how='left')
staff_with_dept = staff_with_dept.merge(ward[['ward_id', 'department_id']], on='ward_id', how='left', suffixes=('', '_ward'))

staff_snapshot = (
    staff_with_dept.groupby(['department_id', 'role'])
    .size()
    .reset_index(name='staff_count')
)
staff_snapshot = staff_snapshot.merge(department[['department_id', 'department_name']], on='department_id', how='left')
staff_snapshot['resource_type'] = 'staff'
staff_snapshot['snapshot_note'] = 'current roster - not date-stamped in source data'

print(staff_snapshot.shape)
staff_snapshot

(22, 6)


,department_id,role,staff_count,department_name,resource_type,snapshot_note
0,1,Nurse,6,Emergency,staff,current roster - not date-stamped in source data
1,1,Technician,10,Emergency,staff,current roster - not date-stamped in source data
2,2,Nurse,11,Internal Medicine,staff,current roster - not date-stamped in source data
3,2,Technician,12,Internal Medicine,staff,current roster - not date-stamped in source data
4,3,Nurse,14,Surgery,staff,current roster - not date-stamped in source data
5,3,Technician,12,Surgery,staff,current roster - not date-stamped in source data
6,4,Nurse,8,Pediatrics,staff,current roster - not date-stamped in source data
7,4,Technician,4,Pediatrics,staff,current roster - not date-stamped in source data
8,5,Nurse,11,Orthopedics,staff,current roster - not date-stamped in source data
9,5,Technician,7,Orthopedics,staff,current roster - not date-stamped in source data


In [7]:
resource_utilization_beds.to_csv(f"{PROCESSED_DIR}/resource_utilization_dataset.csv", index=False)
staff_snapshot.to_csv(f"{PROCESSED_DIR}/resource_utilization_staff_snapshot.csv", index=False)
print("Saved both resource utilization files.")

Saved both resource utilization files.


## 4. Patient Flow Dataset
**Grain: one row = one movement event.**
HMIS has no ward-transfer log (one ward/bed per admission), so flow is modeled as exactly 2 events per admission: `Admission` and `Discharge`. This is documented as a limitation — it does NOT simulate a multi-step ward journey that isn't in the source data.

In [8]:
admission_events = hospital_overview[[
    'admission_id', 'patient_id', 'department_id', 'department_name',
    'ward_id', 'ward_name', 'bed_id', 'admission_date'
]].copy()
admission_events['movement_type'] = 'Admission'
admission_events['movement_date'] = admission_events['admission_date']
admission_events['movement_sequence'] = 1

discharge_events = hospital_overview[[
    'admission_id', 'patient_id', 'department_id', 'department_name',
    'ward_id', 'ward_name', 'bed_id', 'discharge_date'
]].copy()
discharge_events['movement_type'] = 'Discharge'
discharge_events['movement_date'] = discharge_events['discharge_date']
discharge_events['movement_sequence'] = 2

patient_flow = pd.concat([
    admission_events.drop(columns=['admission_date']),
    discharge_events.drop(columns=['discharge_date'])
], ignore_index=True).sort_values(['admission_id', 'movement_sequence'])

patient_flow['year'] = pd.to_datetime(patient_flow['movement_date']).dt.year
patient_flow['month'] = pd.to_datetime(patient_flow['movement_date']).dt.month
patient_flow['day_of_week'] = pd.to_datetime(patient_flow['movement_date']).dt.day_name()

print("patient_flow shape:", patient_flow.shape, "(expected:", hospital_overview.shape[0]*2, ")")
patient_flow.head()

patient_flow shape: (90000, 13) (expected: 90000 )


,admission_id,patient_id,department_id,department_name,ward_id,ward_name,bed_id,movement_type,movement_date,movement_sequence,year,month,day_of_week
257,1,166,2,Internal Medicine,6,Internal Medicine Ward 1,76,Admission,2020-02-25,1,2020,2,Tuesday
45257,1,166,2,Internal Medicine,6,Internal Medicine Ward 1,76,Discharge,2020-02-27,2,2020,2,Thursday
13151,2,8622,5,Orthopedics,21,Orthopedics Ward 1,302,Admission,2022-02-22,1,2022,2,Tuesday
58151,2,8622,5,Orthopedics,21,Orthopedics Ward 1,302,Discharge,2022-03-04,2,2022,3,Friday
36056,3,23976,1,Emergency,2,Emergency Ward 2,11,Admission,2021-02-03,1,2021,2,Wednesday


In [9]:
patient_flow.to_csv(f"{PROCESSED_DIR}/patient_flow_dataset.csv", index=False)
print("Saved:", f"{PROCESSED_DIR}/patient_flow_dataset.csv")

Saved: ../data/processed/patient_flow_dataset.csv


## Summary — Milestone 1 Deliverables
```
data/processed/
├── hospital_overview_dataset.csv          (45,000 rows — one row = one admission)
├── patient_flow_dataset.csv               (90,000 rows — admission + discharge events)
├── department_analytics_dataset.csv       (department + day grain)
├── resource_utilization_dataset.csv       (department + day + bed, daily occupancy)
└── resource_utilization_staff_snapshot.csv (department + role, current snapshot — documented limitation)
```
**Documented limitations (put these in `docs/methodology.md`):**
1. Readmission is derived from HMIS `admission` data alone (same `patient_id`, ≤30 days between discharge and next admission) — NOT cross-matched with the external Readmission Kaggle dataset, since patient IDs are not shared across independently-generated synthetic datasets.
2. Staff Resource Utilization is a snapshot (current roster), not a daily trend — `staff_assignment` has no date field in the source data.
3. Patient Flow is a 2-point journey (Admission → Discharge), not a multi-ward transfer log — HMIS assigns exactly one ward/bed per admission.
4. Beds Management, Readmission, and Inpatient Discharges (SPARCS) datasets remain standalone/supplementary per the mentor's instruction — not merged row-level into the HMIS-based core tables.